In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.classify import NaiveBayesClassifier
from nltk.classify.util import accuracy 
from nltk.corpus import movie_reviews


In [2]:
# Download necessary NLTK data files
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('movie_reviews')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rohan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rohan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Rohan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Rohan\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

In [3]:

# Text preprocessing function
def preprocess_text(text):
    # Tokenize the text
    tokens = word_tokenize(text)
    # Convert to lowercase
    tokens = [word.lower() for word in tokens]
    # Remove punctuation and non-alphabetic characters
    tokens = [word for word in tokens if word.isalpha()]
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatize tokens
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens


In [ ]:
# Feature extraction function
def extract_features(words):
    return {word: True for word in words}


In [5]:

# Load movie reviews dataset
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]
# Preprocess the documents
documents = [(preprocess_text(' '.join(doc)), category) for doc, category in documents]


In [6]:

# Split into training and testing sets
train_set, test_set = documents[:1600], documents[1600:]

# Extract features
train_features = [(extract_features(doc), category) for doc, category in train_set]
test_features = [(extract_features(doc), category) for doc, category in test_set]

# Train Naive Bayes classifier
classifier = NaiveBayesClassifier.train(train_features)

# Evaluate the classifier
accuracy = accuracy(classifier, test_features)
print(f'Test accuracy: {accuracy * 100:.2f}%')

# Show most informative features
classifier.show_most_informative_features(10)


Test accuracy: 97.00%
Most Informative Features
               ludicrous = True              neg : pos    =     15.4 : 1.0
                  avoids = True              pos : neg    =     12.8 : 1.0
                  hatred = True              pos : neg    =     12.8 : 1.0
               stupidity = True              neg : pos    =     12.6 : 1.0
              astounding = True              pos : neg    =     11.7 : 1.0
                    deft = True              pos : neg    =     11.7 : 1.0
             fascination = True              pos : neg    =     11.7 : 1.0
               insulting = True              neg : pos    =     11.0 : 1.0
             outstanding = True              pos : neg    =     10.8 : 1.0
                  annual = True              pos : neg    =     10.5 : 1.0


In [11]:
def classify_text(text):
    processed_text = preprocess_text(text)
    features = extract_features(processed_text)
    return classifier.classify(features)    

input_text = "I loved the movie, it was fantastic!"
print(f'The sentiment of the input text is: {classify_text(input_text)}')

The sentiment of the input text is: pos


In [12]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Load the data from the Excel file

df = pd.read_excel('spacydata.xlsx')

# Display the first few rows of the dataframe to understand its structure
print(df.head())

# Define the text preprocessing function using spaCy
def preprocess_text_spacy(text):
    doc = nlp(text)
    tokens = [token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop]
    return ' '.join(tokens)

# Apply preprocessing to the text data
df['text'] = df['text'].apply(preprocess_text_spacy)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

# Build a pipeline with CountVectorizer and Naive Bayes classifier
model = make_pipeline(CountVectorizer(), MultinomialNB())

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy * 100:.2f}%')


                                                text label
0  films adapted from comic books have had plenty...   pos
1  for starters , it was created by alan moore ( ...   pos
2  to say moore and campbell thoroughly researche...   pos
3  the book ( or " graphic novel , " if you will ...   pos
4  in other words , don't dismiss this film becau...   pos
Test accuracy: 100.00%
